# 💬 生成式AI應用開發｜第05週
## 對話狀態、Streaming 與聊天機器人

> **執行環境：Google Colab** ｜ **延續第 3、4 週的 OpenAI Responses API**

前幾週的呼叫都是「問一次、答一次」。本週讓程式**記住上下文**，
並讓回覆像打字一樣**逐字出現（Streaming）**，最後組成一個簡易聊天機器人。你會學到：

1. 為什麼單次呼叫「沒有記憶」
2. 兩種多輪對話做法：`previous_response_id` 與自己維護 messages
3. 用 `ChatSession` 管理對話狀態與簡易記憶
4. Streaming 逐字輸出
5. 上下文長度與成本控制
6. 組合成可多輪對話 + 串流的聊天機器人

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  完成一個具備多輪對話記憶與串流輸出的簡易聊天機器人。
</div>


> **學生版**：本檔保留實作與練習的 TODO 骨架，請依照註解完成程式。
> 需要執行 OpenAI API 的 cells 會產生費用，請依老師指示執行。

## 🎯 本週學習目標

| # | 能力 | 對應後續課程 |
|---|------|-------------|
| 1 | 理解「單次呼叫沒有記憶」 | 所有對話式應用 |
| 2 | 用 `previous_response_id` 延續對話 | 快速多輪對話 |
| 3 | 自己維護 messages history | 可控上下文、跨平台 |
| 4 | 用 `ChatSession` 管理對話狀態 | 聊天機器人、客服助手 |
| 5 | Streaming 逐字輸出 | 提升使用體驗 |
| 6 | 上下文長度與成本控制 | 控制 API 花費 |


---
## 🧰 第一節：環境準備（延續第 3、4 週）

沿用先前的環境設定與 `ask_ai_safe()`。Colab 每次重開 runtime 都要重新安裝與設定金鑰。


In [ ]:
# 安裝 OpenAI Python SDK
!pip install openai --quiet

import os
import json
from openai import OpenAI

print("套件與模組準備完成")


### 🔐 設定 API Key

作法與第 3 週相同：在 Colab 左側 **Secrets** 面板新增 `OPENAI_API_KEY`，並打開存取權限。


In [ ]:
# 從 Colab Secrets 讀取 API key，並轉成環境變數
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("已從 Colab Secrets 載入 API key")
except Exception as e:
    print("非 Colab 環境或尚未設定 Secrets：", e)
    print("請改用環境變數或 .env 設定 OPENAI_API_KEY")


In [ ]:
# 建立 client、選模型，並沿用第 3 週的安全呼叫函式
client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")


def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    """單次呼叫 Responses API，回傳 (是否成功, 文字)。"""
    try:
        response = client.responses.create(
            model=model, instructions=role, input=user_input
        )
        return True, response.output_text
    except Exception as e:
        return False, f"API 呼叫失敗：{e}"


print(f"準備完成，使用模型：{MODEL}")


---
## 🧠 第二節：單次呼叫「沒有記憶」

每次 `ask_ai_safe()` 都是獨立呼叫，模型不會記得上一次講過什麼。先觀察這個現象。


In [ ]:
# 示範：兩次獨立呼叫，模型記不得第一句的名字
ok, a1 = ask_ai_safe("我叫小明，正在學 Python。")
print("第一輪：", a1)

ok, a2 = ask_ai_safe("我剛剛說我叫什麼名字？")
print("第二輪：", a2)   # 模型通常答不出來，因為它看不到上一輪


---
## 🔁 第三節：方法 A —— previous_response_id

Responses API 內建的多輪對話：把上一輪的 `response.id` 傳給下一輪的 `previous_response_id`，
平台會自動幫你接上上下文，你只要送**新的一句**。


In [ ]:
# 示範：用 previous_response_id 延續對話
r1 = client.responses.create(model=MODEL, input="我叫小明，正在學 Python。")
print("第一輪：", r1.output_text)

r2 = client.responses.create(
    model=MODEL,
    previous_response_id=r1.id,   # 接上一輪
    input="我剛剛說我叫什麼名字？"
)
print("第二輪：", r2.output_text)   # 這次應該記得「小明」


---
## 🧱 第四節：方法 B —— 自己維護 messages history

另一種做法是自己把每一輪的 user / assistant 訊息**累積成清單**，每次整包送出。
這種方式最通用（跨平台都適用），也讓你能**完全掌控**要保留哪些上下文。


In [ ]:
# 示範：手動累積對話歷史
system = "你是一位友善的助理，用繁體中文簡潔回答。"
history = []   # 只存 user / assistant

def chat_once(user_msg):
    # 組成這次要送出的完整訊息：system + 過去歷史 + 這次的 user
    msgs = [{"role": "developer", "content": system}] + history + \
           [{"role": "user", "content": user_msg}]
    resp = client.responses.create(model=MODEL, input=msgs)
    answer = resp.output_text
    # 把這一輪的問與答存回歷史
    history.append({"role": "user", "content": user_msg})
    history.append({"role": "assistant", "content": answer})
    return answer

print(chat_once("我叫小明，正在學 Python。"))
print(chat_once("我剛剛說我叫什麼名字？"))   # 應該記得
print("目前歷史長度（則）：", len(history))


### 🧭 4-1 兩種方法怎麼選？

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>previous_response_id vs 自己維護 messages</b></font><br>
  • <b>previous_response_id</b>：最省事，平台幫你記；但上下文由平台掌控，較難自訂裁切。<br>
  • <b>自己維護 messages</b>：程式碼多一點，但可完全控制保留哪些內容、方便做成本控制，且概念可直接套到 Gemini / Claude。<br>
  本週後面採用「<b>自己維護 messages</b>」，因為它最適合做聊天機器人與成本控制。
</div>


---
## 📦 第五節：用 ChatSession 管理對話狀態

把「system 設定 + 歷史累積 + 自動裁切」包成一個類別，之後聊天機器人直接用它。


In [ ]:
class ChatSession:
    def __init__(self, system="你是一位友善的助理，用繁體中文回答。", model=MODEL, max_turns=6):
        self.system = system
        self.model = model
        self.max_turns = max_turns
        self.history = []

    def _build_input(self, user_msg):
        # TODO 1：回傳 [system] + self.history + [這次的 user 訊息]
        return [{"role": "user", "content": user_msg}]

    def _trim(self):
        # TODO 2：只保留最近 max_turns 輪（提示：max_msgs = max_turns * 2）
        pass

    def send(self, user_msg):
        # TODO 3：呼叫 client.responses.create(model=..., input=self._build_input(user_msg))
        # TODO 4：把 user 與 assistant 訊息存回 self.history，呼叫 self._trim()，回傳答案
        return "尚未完成 ChatSession.send()"

    def reset(self):
        self.history = []


chat = ChatSession()
print(chat.send("我叫小明，正在學 Python。"))
print(chat.send("我剛剛說我叫什麼名字？"))


---
## ⚡ 第六節：Streaming 逐字輸出

一次等整段回覆才顯示，體感很慢。Streaming 讓文字**一邊產生一邊顯示**。
Responses API 只要在呼叫時加上 `stream=True`，就會回傳一串「事件」，
我們取出型別為 `response.output_text.delta` 的事件，把 `delta`（新增的片段）印出來。


In [ ]:
# 示範：串流輸出，文字會逐段出現
stream = client.responses.create(
    model=MODEL,
    instructions="你是一位老師，用繁體中文回答。",
    input="請用 100 字介紹什麼是 Streaming 輸出。",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)   # 逐段印出，不換行
print()   # 最後換行


### 📡 6-1 把 Streaming 封裝成函式

實務上會把串流迴圈包起來，並在印出的同時累積完整文字，方便存回對話歷史。


In [ ]:
def stream_reply(user_input, role="你是一位有幫助的助理", model=MODEL):
    full = ""
    # TODO 1：呼叫 client.responses.create(..., stream=True)
    # TODO 2：用 for 迴圈讀事件，型別為 "response.output_text.delta" 時
    #        print(event.delta, end="", flush=True) 並把 delta 累加到 full
    print("尚未完成 stream_reply()")
    return full


text = stream_reply("請用三點說明多輪對話為什麼需要保存歷史。")
print("---")
print("完整回覆長度（字）：", len(text))


---
## 💰 第七節：上下文長度與成本控制

對話越長，每次送出的歷史就越多，token 與費用也越高。
常見策略：**只保留最近 N 輪**（`ChatSession` 已內建 `_trim`），必要時再加上摘要舊對話。


In [ ]:
# 估算對話歷史的規模（以字數粗估；真正的 token 數以 response.usage 為準）
def history_size(history):
    chars = sum(len(m["content"]) for m in history)
    return {"則數": len(history), "總字數": chars}


# 觀察 max_turns 對歷史長度的影響
short_chat = ChatSession(max_turns=2)   # 只記最近 2 輪
for msg in ["我叫小明。", "我在學 Python。", "我喜歡打籃球。", "我剛剛第一句說什麼？"]:
    short_chat.send(msg)

print("最近保留的歷史：", history_size(short_chat.history))
print("提醒：max_turns 太小，模型會忘記較早的內容；太大則成本上升。")


### 🧮 7-1 成本控制小抄

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>控制對話成本的幾個做法</b></font><br>
  1. 只保留最近 N 輪（<code>max_turns</code>）。<br>
  2. 舊對話先「摘要」再保留，取代逐字歷史。<br>
  3. 課堂一律用 mini 等級模型。<br>
  4. 用 <code>response.usage</code> 追蹤實際 token，定期檢查花費。
</div>


---
## 🤖 第八節：組合成簡易聊天機器人

把「維護歷史 + 裁切 + Streaming」組合起來，就是一個可多輪對話、逐字回覆的聊天機器人。
在 Colab 中，`input()` 會跳出輸入框；輸入 `quit` 結束。


In [ ]:
def chat_bot(system="你是課程助教，用繁體中文、簡潔回答。", max_turns=6):
    history = []
    print("聊天機器人啟動，輸入 quit 結束。")
    while True:
        user_msg = input("你：")
        if user_msg.strip().lower() in ("quit", "exit"):
            print("助教：掰掰！")
            break
        # TODO 1：組成 msgs = [system] + history + [user_msg]
        # TODO 2：用 stream=True 逐字印出回覆，並累加到 full
        # TODO 3：把 user 與 assistant 存回 history
        # TODO 4：只保留最近 max_turns 輪（history = history[-max_turns * 2:]）
        full = ""
        print("助教：（尚未完成 chat_bot）")


# 完成後，在 Colab 取消下一行註解開始對話：
# chat_bot()


---
## 🧪 第九節：課堂練習

### 📝 練習 A：主題聊天機器人

用 `ChatSession` 做一個有固定角色的多輪聊天機器人，例如「英文練習夥伴」：
設定 `system` 讓它用簡單英文回話並溫和糾正，連續對話 3 輪以上，確認它記得前面內容。

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
# 練習 A：主題聊天機器人
# TODO 1：建立一個 ChatSession，設定符合你選定角色的 system（例如英文練習夥伴）
partner = None

# TODO 2：連續 send() 至少 3 次，確認它記得前面內容
# print(partner.send("..."))
print("尚未完成練習 A")


---
### 🔁 練習 B：上下文長度實驗

建立一個 `max_turns=1` 的 `ChatSession`，先告訴它你的名字，
中間再聊兩三句，最後問「我叫什麼名字？」，觀察它是否已經忘記。想一想為什麼。


In [ ]:
# 練習 B：觀察 max_turns 太小造成遺忘
# TODO 1：建立 max_turns=1 的 ChatSession
forgetful = None

# TODO 2：先說名字，再聊 2~3 句，最後問「我叫什麼名字？」
# TODO 3：印出結果與 forgetful.history，想想為什麼會忘記
print("尚未完成練習 B")


---
### 💰 練習 C（選做）：累計 token 與估算成本

改寫 `ChatSession.send()`，讓它同時回傳 `response.usage`，
在多輪對話中累加 input / output token，最後印出整場對話的 token 總量。


In [ ]:
# 練習 C（選做）：累計整場對話的 token
def get_usage_value(usage, field_name):
    if usage is None:
        return 0
    if isinstance(usage, dict):
        return usage.get(field_name, 0) or 0
    return getattr(usage, field_name, 0) or 0


# TODO 1：跑一段多輪對話，每輪從 resp.usage 讀 input_tokens / output_tokens
# TODO 2：用 total_in、total_out 累加，最後印出整場總量
total_in, total_out = 0, 0
print("尚未完成練習 C")


---
## ✅ 本週小結

| 技能 | 本週學了什麼 | 下週用在哪裡 |
|------|-------------|-------------|
| 沒有記憶 | 單次呼叫彼此獨立 | 理解為何要保存歷史 |
| previous_response_id | 平台幫你接上一輪 | 快速多輪對話 |
| 維護 messages | 自己累積 user/assistant | 可控上下文、跨平台 |
| ChatSession | 管理 system、歷史、裁切 | 聊天機器人、客服助手 |
| Streaming | `stream=True` 逐字輸出 | Week 7 Streamlit 串流顯示 |
| 成本控制 | 只保留最近 N 輪、追蹤 usage | 控制 API 花費 |

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>下週預告</b></font><br>
  Week 6 進入 Structured Outputs 與資料抽取，把模型回覆變成可驗證的 JSON 結構化資料。
</div>
